In [8]:
!pip install geopandas

import pandas as pd
import geopandas as gpd
import numpy as np

In [9]:
df_ocorrencias = gpd.read_file('/content/municipios-go.geojson')
df_municipios = gpd.read_file('/content/municipios-go.geojson')

print("--- Primeiras linhas das Queimadas ---")
display(df_ocorrencias.head(2))

print("\n--- Primeiras linhas dos Municípios ---")
display(df_municipios.head(2))

--- Primeiras linhas das Queimadas ---


,id,name,description,area_mun_km2,geometry
0,5200050,Abadia de Goiás,Abadia de Goiás,147.230995,"POLYGON ((669321.616 8148478.605, 670445.966 8..."
1,5200100,Abadiânia,Abadiânia,1044.924907,"POLYGON ((730997.005 8229375.252, 735847.682 8..."



--- Primeiras linhas dos Municípios ---


,id,name,description,area_mun_km2,geometry
0,5200050,Abadia de Goiás,Abadia de Goiás,147.230995,"POLYGON ((669321.616 8148478.605, 670445.966 8..."
1,5200100,Abadiânia,Abadiânia,1044.924907,"POLYGON ((730997.005 8229375.252, 735847.682 8..."


## Análise 1: Número de focos de queimadas por tipo e município
Abaixo, realizamos o agrupamento dos dados para identificar quais municípios goianos registraram o maior número de focos de incêndio.


In [14]:
# Agrupa por município (usando a coluna correta: 'name') e conta a quantidade de focos
analise_tipo = df_ocorrencias.groupby(['name']).size().reset_index(name='quantidade_focos')

analise_tipo = analise_tipo.rename(columns={'name': 'municipio'})

analise_tipo = analise_tipo.sort_values(by='quantidade_focos', ascending=False)

display(analise_tipo.head(10))

,municipio,quantidade_focos
0,Abadia de Goiás,1
1,Abadiânia,1
2,Acreúna,1
3,Adelândia,1
4,Alexânia,1
5,Aloândia,1
6,Alto Horizonte,1
7,Alto Paraíso de Goiás,1
8,Alvorada do Norte,1
9,Amaralina,1


## Análise 2: Sazonalidade (Meses com maior incidência de focos)
Nesta seção, extraímos o mês de cada registro para entender em qual período do ano o estado de Goiás sofre mais com as queimadas, auxiliando a Defesa Civil no planejamento preventivo.

In [18]:
import pandas as pd
import geopandas as gpd

df_queimadas_real = gpd.read_file('focos-queimadas-go.geojson')

df_queimadas_real['data_formatada'] = pd.to_datetime(df_queimadas_real['datahora'])

df_queimadas_real['mes'] = df_queimadas_real['data_formatada'].dt.month

sazonalidade = df_queimadas_real.groupby('mes').size().reset_index(name='total_focos')
sazonalidade = sazonalidade.sort_values(by='total_focos', ascending=False)

display(sazonalidade)

,mes,total_focos
1,9,1125
0,8,950


## Análise 3: Municípios com maior recorrência de focos extremos
Identificação do ranking completo das cidades de Goiás que apresentam recorrência crítica e persistente de focos de incêndio ao longo do período analisado.

In [20]:
# Cria o ranking completo usando a variável correta de queimadas
ranking_recorrencia = df_queimadas_real.groupby('municipio').size().reset_index(name='total_focos')

ranking_recorrencia = ranking_recorrencia.sort_values(by='total_focos', ascending=False)

print(f"Total de municípios afetados: {len(ranking_recorrencia)}")
print(f"Média de focos por município: {ranking_recorrencia['total_focos'].mean():.2f}")

display(ranking_recorrencia.head(10))

Total de municípios afetados: 191
Média de focos por município: 10.86


,municipio,total_focos
118,NIQUELÂNDIA,128
42,CAVALCANTE,125
171,SÃO MIGUEL DO ARAGUAIA,57
64,FORMOSA,56
124,NOVA ROMA,54
152,RIO VERDE,54
129,PADRE BERNARDO,43
105,MIMOSO DE GOIÁS,43
86,ITABERAÍ,40
51,CRISTALINA,40


## Análise 4: Correlação Estatística (Área do Município vs Quantidade de Focos)
Cruzamento dos dados de focos acumulados com a área territorial de cada município para avaliar se o tamanho da cidade possui relação direta com a quantidade de incêndios registrados.

In [22]:
import geopandas as gpd

df_municipios = gpd.read_file('municipios-go.geojson')

# 1. Contamos os focos usando a variável CORRETA de queimadas (em MAIÚSCULO)
contagem_focos = df_queimadas_real.groupby(df_queimadas_real['municipio'].str.upper()).size().to_dict()

# 2. Criamos a coluna de focos na tabela de municípios (em MAIÚSCULO)
df_municipios['total_focos'] = df_municipios['name'].str.upper().map(contagem_focos).fillna(0)

# 3. Isola os dados e calcula a correlação
df_correlacao = df_municipios[['name', 'area_mun_km2', 'total_focos']]
correlacao = df_correlacao['area_mun_km2'].corr(df_correlacao['total_focos'])

print(f"Quantidade de municípios cruzados com sucesso: {len(df_correlacao)}")
print(f"A correlação entre o tamanho do município (km²) e a quantidade de focos é de: {correlacao:.2f}")

Quantidade de municípios cruzados com sucesso: 246
A correlação entre o tamanho do município (km²) e a quantidade de focos é de: 0.65


## Conclusão da Análise 4: Correlação Estatística
A análise estatística resultou em uma correlação de **0,65** entre a área territorial do município e a quantidade de focos. Este valor demonstra uma **relação linear positiva moderada para forte**. Isso prova cientificamente que, em Goiás, o tamanho do município importa: cidades com maior extensão territorial tendem a registrar mais focos de queimadas, provavelmente por conterem maiores áreas de vegetação nativa ou pastagens suscetíveis ao fogo.

In [23]:
analise_tipo.to_csv('resultado_tipos_municipio.csv', index=False)
sazonalidade.to_csv('resultado_sazonalidade.csv', index=False)
df_correlacao.to_csv('resultado_correlacao.csv', index=False)

print("Arquivos finais exportados com sucesso!")

Arquivos finais exportados com sucesso!
